<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 16px; margin-bottom: 24px; border: 1px solid #e94560;">
  <h1 style="color: #e94560; font-family: 'Segoe UI', sans-serif; font-size: 2.4em; margin: 0 0 8px 0; text-align: center;">&#x1F680; Redrob Ranker &#8212; Judge Sandbox</h1>
  <p style="color: #a8b2d8; font-family: 'Segoe UI', sans-serif; font-size: 1.1em; text-align: center; margin: 0;">Team: <strong style='color:#e2e8f0;'>UniverseX</strong> &nbsp;|&nbsp; Upload any small <code>.jsonl</code> candidate file and watch the full pipeline run live.</p>
  <hr style='border: none; border-top: 1px solid #e9456044; margin: 20px 0;'/>
  <p style="color: #8892b0; font-size: 0.95em; text-align: center; margin: 0;">No local setup required &nbsp;&middot;&nbsp; Runs entirely in Colab &nbsp;&middot;&nbsp; ~3 min end-to-end</p>
</div>

## What This Sandbox Does

This notebook reproduces the **exact same pipeline** used in our final submission:

| Step | What happens |
|------|--------------|
| **1. Setup** | Install dependencies |
| **2. Load Modules** | Inline source code from `src/` |
| **3. Upload** | Upload any small `candidates.jsonl` file |
| **4. Preprocess** | Parse profiles, skills, career history, signals |
| **5. Features** | 60+ features: career stability, AI skill score, product-company ratio |
| **6. Semantic** | Embed candidates + JD using `BAAI/bge-small-en-v1.5` |
| **7. Behavior** | Availability, trust, engagement, recruiter interest |
| **8. Ranking** | Weighted final score, rank candidates |
| **9. Download** | Generate + download `UniverseX_sandbox_submission.csv` |

> **Quick note**: Upload the small `candidte.jsonl` (2 records) from the repo root, or any valid subset. The pipeline is identical regardless of size.


---
## Step 1 — Install Dependencies

In [ ]:
# STEP 1: Install all required packages (takes ~60 seconds)
print('Installing dependencies...')
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers>=2.2.2',
    'pandas>=2.0.0',
    'numpy>=1.22.0',
    'scikit-learn>=1.0.0',
    'pyarrow>=10.0.0',
    'tqdm>=4.64.0'
])
print('Dependencies installed successfully!')

---
## Step 2 — Load Source Modules (Inlined from src/)

In [ ]:
# =================================================================
# 2a: Preprocessing  (mirrors src/loader.py + src/preprocessing.py)
# =================================================================
import json, re
import pandas as pd
import numpy as np

def load_candidates_from_string(text):
    candidates = []
    for line in text.splitlines():
        if line.strip():
            candidates.append(json.loads(line))
    return candidates

def build_profiles_df(candidates):
    return pd.DataFrame([{"candidate_id": c["candidate_id"], **c["profile"]} for c in candidates])

def build_skill_df(candidates):
    rows = []
    for c in candidates:
        for s in c["skills"]:
            rows.append({"candidate_id": c["candidate_id"], "skill_name": s["name"],
                         "proficiency": s["proficiency"], "endorsements": s["endorsements"],
                         "duration_months": s.get("duration_months", 0)})
    return pd.DataFrame(rows)

def build_career_df(candidates):
    rows = []
    for c in candidates:
        for j in c["career_history"]:
            rows.append({"candidate_id": c["candidate_id"], "company": j["company"],
                         "title": j["title"], "industry": j["industry"],
                         "company_size": j["company_size"], "start_date": j["start_date"],
                         "end_date": j["end_date"], "duration_months": j["duration_months"],
                         "is_current": j["is_current"], "description": j["description"]})
    return pd.DataFrame(rows)

def build_signals_df(candidates):
    return pd.DataFrame([{"candidate_id": c["candidate_id"], **c["redrob_signals"]} for c in candidates])

def prepare_data(candidates):
    return {
        "profiles": build_profiles_df(candidates),
        "skills":   build_skill_df(candidates),
        "career":   build_career_df(candidates),
        "signals":  build_signals_df(candidates)
    }

print('Preprocessing module loaded.')

In [ ]:
# =================================================================
# 2b: Feature Engineering  (mirrors src/feature_engineering.py)
# =================================================================

def build_experience_features(p): return p[["candidate_id","years_of_experience"]].copy()
def build_title_features(p):      return p[["candidate_id","current_title"]].copy()

def build_career_stability_features(career_df):
    return (career_df.groupby("candidate_id")
            .agg(total_jobs=("company","count"), unique_companies=("company","nunique"),
                 avg_job_duration=("duration_months","mean"),
                 shortest_job=("duration_months","min"), longest_job=("duration_months","max"))
            .reset_index())

def build_product_company_features(career_df):
    PROD = {"Meta","Google","Flipkart","Swiggy","Razorpay","CRED","Meesho","Zoho","Freshworks","Nykaa","Paytm"}
    SERV = {"Infosys","TCS","Wipro","Accenture","Capgemini","Cognizant","HCL","Tech Mahindra","Mindtree","Mphasis"}
    df = career_df.copy()
    df["is_product_company"] = df["company"].isin(PROD).astype(int)
    df["is_service_company"] = df["company"].isin(SERV).astype(int)
    return (df.groupby("candidate_id")
            .agg(product_company_ratio=("is_product_company","mean"),
                 service_company_ratio=("is_service_company","mean"),
                 product_company_count=("is_product_company","sum"),
                 service_company_count=("is_service_company","sum"))
            .reset_index())

def build_career_description_features(career_df):
    KW = {
        "retrieval":      ["retrieval","information retrieval","semantic search"],
        "ranking":        ["ranking","learning to rank","ltr"],
        "recommendation": ["recommendation","recommendation system","recommender"],
        "embeddings":     ["embedding","embeddings","sentence transformer"],
        "vector_db":      ["faiss","pinecone","milvus","qdrant","weaviate"],
        "evaluation":     ["ndcg","mrr","precision","recall","offline evaluation","a/b testing"],
        "production":     ["production","deployment","deployed","monitoring"]
    }
    df = career_df.copy()
    df["description"] = df["description"].fillna("").str.lower()
    rows = []
    for cid, grp in df.groupby("candidate_id"):
        text = " ".join(grp["description"])
        row = {"candidate_id": cid}
        for feat, words in KW.items():
            row[f"{feat}_mentions"] = sum(len(re.findall(re.escape(w), text)) for w in words)
        rows.append(row)
    return pd.DataFrame(rows)

def build_python_features(skill_df):
    PM = {"beginner":1,"intermediate":2,"advanced":3,"expert":4}
    df = skill_df[skill_df["skill_name"].str.lower()=="python"].copy()
    if df.empty:
        return pd.DataFrame(columns=["candidate_id","python_duration","python_endorsements","python_proficiency"])
    df["python_proficiency"] = df["proficiency"].str.lower().map(PM)
    return (df.groupby("candidate_id")
            .agg(python_duration=("duration_months","max"),
                 python_endorsements=("endorsements","max"),
                 python_proficiency=("python_proficiency","max"))
            .reset_index())

def build_embedding_features(skill_df):
    ES = {"Embeddings","Sentence Transformers","LoRA","QLoRA","PEFT"}
    df = skill_df[skill_df["skill_name"].isin(ES)].copy()
    if df.empty:
        return pd.DataFrame(columns=["candidate_id","embedding_skill_count","embedding_duration","embedding_endorsements"])
    return (df.groupby("candidate_id")
            .agg(embedding_skill_count=("skill_name","count"),
                 embedding_duration=("duration_months","sum"),
                 embedding_endorsements=("endorsements","sum"))
            .reset_index())

def build_vector_database_features(skill_df):
    VD = {"FAISS","Pinecone","Milvus","Qdrant","Weaviate"}
    df = skill_df[skill_df["skill_name"].isin(VD)].copy()
    if df.empty:
        return pd.DataFrame(columns=["candidate_id","vector_db_skill_count","vector_db_duration","vector_db_endorsements"])
    return (df.groupby("candidate_id")
            .agg(vector_db_skill_count=("skill_name","count"),
                 vector_db_duration=("duration_months","sum"),
                 vector_db_endorsements=("endorsements","sum"))
            .reset_index())

def build_skill_summary(skill_df):
    return (skill_df.groupby("candidate_id")
            .agg(total_skills=("skill_name","count"),
                 avg_skill_duration=("duration_months","mean"),
                 total_endorsements=("endorsements","sum"))
            .reset_index())

def build_behavior_features(signals_df):
    df = signals_df.copy()
    for col in ["open_to_work_flag","willing_to_relocate","verified_email","verified_phone","linkedin_connected"]:
        if col in df.columns: df[col] = df[col].astype(int)
    wm = {"onsite":0,"hybrid":1,"flexible":2,"remote":3}
    df["preferred_work_mode"] = df["preferred_work_mode"].str.lower().map(wm).fillna(-1).astype(int)
    df["expected_salary_min"] = df["expected_salary_range_inr_lpa"].apply(lambda x: x.get("min",np.nan) if isinstance(x,dict) else np.nan)
    df["expected_salary_max"] = df["expected_salary_range_inr_lpa"].apply(lambda x: x.get("max",np.nan) if isinstance(x,dict) else np.nan)
    df["num_skill_assessments"] = df["skill_assessment_scores"].apply(lambda x: len(x) if isinstance(x,dict) else 0)
    keep = ["candidate_id","profile_completeness_score","open_to_work_flag","profile_views_received_30d",
            "applications_submitted_30d","recruiter_response_rate","avg_response_time_hours","connection_count",
            "endorsements_received","notice_period_days","preferred_work_mode","willing_to_relocate",
            "github_activity_score","search_appearance_30d","saved_by_recruiters_30d","interview_completion_rate",
            "offer_acceptance_rate","verified_email","verified_phone","linkedin_connected",
            "expected_salary_min","expected_salary_max","num_skill_assessments"]
    return df[[c for c in keep if c in df.columns]].copy()

def _ai_skill_score(f):
    d = f.copy()
    d["ai_skill_score"] = (3*d.get("python_proficiency",0)+2*d.get("python_endorsements",0)/10+
        4*d.get("embedding_skill_count",0)+3*d.get("vector_db_skill_count",0)+
        2*d.get("retrieval_mentions",0)+2*d.get("ranking_mentions",0)+
        2*d.get("recommendation_mentions",0)+2*d.get("embeddings_mentions",0)+2*d.get("vector_db_mentions",0))
    return d[["candidate_id","ai_skill_score"]]

def _production_score(f):
    d = f.copy()
    d["production_score"] = (2*d.get("years_of_experience",0)+2*d.get("avg_job_duration",0)/12+
        4*d.get("production_mentions",0)+3*d.get("product_company_count",0))
    return d[["candidate_id","production_score"]]

def _career_quality_score(f):
    d = f.copy()
    d["career_quality_score"] = (2*d.get("avg_job_duration",0)/12+2*d.get("product_company_ratio",0)+
        d.get("unique_companies",0)-0.5*d.get("total_jobs",0))
    return d[["candidate_id","career_quality_score"]]

def _recruiter_score(f):
    d = f.copy()
    d["recruiter_score"] = (4*d.get("open_to_work_flag",0)+4*d.get("recruiter_response_rate",0)+
        2*d.get("profile_views_received_30d",0)/10+2*d.get("saved_by_recruiters_30d",0)-
        d.get("notice_period_days",0)/30-d.get("avg_response_time_hours",0)/100)
    return d[["candidate_id","recruiter_score"]]

def _trust_score_fe(f):
    d = f.copy()
    d["trust_score"] = (d.get("verified_email",0)+d.get("verified_phone",0)+d.get("linkedin_connected",0)+
        d.get("profile_completeness_score",0)/20+5*d.get("interview_completion_rate",0))
    return d[["candidate_id","trust_score"]]

def build_all_features(profiles_df, career_df, skill_df, signals_df):
    f = build_experience_features(profiles_df)
    for fn, src in [
        (build_title_features, profiles_df),
        (build_career_stability_features, career_df),
        (build_product_company_features, career_df),
        (build_career_description_features, career_df),
        (build_python_features, skill_df),
        (build_embedding_features, skill_df),
        (build_vector_database_features, skill_df),
        (build_skill_summary, skill_df),
        (build_behavior_features, signals_df),
    ]:
        f = f.merge(fn(src), on="candidate_id", how="left")
    f = f.fillna(0)
    for sfn in [_ai_skill_score, _production_score, _career_quality_score, _recruiter_score, _trust_score_fe]:
        f = f.merge(sfn(f), on="candidate_id")
    return f.fillna(0)

print('Feature Engineering module loaded.')

In [ ]:
# =================================================================
# 2c: Behavior Scoring  (mirrors src/behavior_scoring.py)
# =================================================================

def _mm(s):
    s = s.fillna(0)
    return pd.Series(0.0, index=s.index) if s.max()==s.min() else (s-s.min())/(s.max()-s.min())

def build_availability_score(df):
    d = df.copy()
    d["open_to_work"] = d["open_to_work_flag"].astype(int)
    avail = 0.35*d["open_to_work"]+0.30*d["recruiter_response_rate"].clip(0,1)+(0.20*(1-_mm(d["avg_response_time_hours"])))+(0.15*(1-_mm(d["notice_period_days"])))
    return pd.DataFrame({"candidate_id": d["candidate_id"], "availability_score": avail})

def build_recruiter_interest_score(df):
    d = df.copy()
    ri = (0.25*_mm(d["profile_views_received_30d"])+0.25*_mm(d["search_appearance_30d"])+
          0.20*_mm(d["saved_by_recruiters_30d"])+0.15*_mm(d["endorsements_received"])+0.15*_mm(d["connection_count"]))
    return pd.DataFrame({"candidate_id": d["candidate_id"], "recruiter_interest_score": ri})

def build_trust_score_bs(df):
    d = df.copy()
    for col in ["verified_email","verified_phone","linkedin_connected"]: d[col] = d[col].astype(int)
    trust = 0.40*(d["profile_completeness_score"]/100)+0.20*d["verified_email"]+0.20*d["verified_phone"]+0.20*d["linkedin_connected"]
    return pd.DataFrame({"candidate_id": d["candidate_id"], "trust_score": trust})

def build_engagement_score(df):
    d = df.copy()
    github = (d["github_activity_score"].replace(-1,0)/100).clip(0,1)
    assessments = _mm(d["skill_assessment_scores"].apply(lambda x: len(x) if isinstance(x,dict) else 0))
    eng = 0.35*d["interview_completion_rate"].clip(0,1)+0.30*d["offer_acceptance_rate"].replace(-1,0).clip(0,1)+0.20*github+0.15*assessments
    return pd.DataFrame({"candidate_id": d["candidate_id"], "engagement_score": eng})

def build_behavior_score(signals_df):
    avail = build_availability_score(signals_df)
    recr  = build_recruiter_interest_score(signals_df)
    trust = build_trust_score_bs(signals_df)
    eng   = build_engagement_score(signals_df)
    beh   = avail.merge(recr,on="candidate_id").merge(trust,on="candidate_id").merge(eng,on="candidate_id")
    beh["behavior_score"] = (0.40*beh["availability_score"]+0.15*beh["recruiter_interest_score"]+
                             0.20*beh["trust_score"]+0.25*beh["engagement_score"])
    return beh

print('Behavior Scoring module loaded.')

In [ ]:
# =================================================================
# 2d: Ranking + Embedding  (mirrors src/ranking.py + src/embedding.py)
# =================================================================
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# --- Job Description (same as used in submission) ---
JOB_DESCRIPTION = """
Senior AI Engineer - Founding Team

We are building the core AI/ML infrastructure for Redrob's candidate intelligence platform.
You will own the ranking and retrieval systems that power our recruiter product end-to-end.

Responsibilities:
- Design and deploy production-grade embedding pipelines and vector search systems.
- Build and iterate on learning-to-rank (LTR) models, semantic retrieval, recommendation systems.
- Instrument models with offline (NDCG, MRR) and online (A/B test) evaluation frameworks.
- Work directly with recruiters and product to translate requirements into ML features.

Requirements:
- 5-9 years of hands-on ML/AI experience, preferably at a product company.
- Deep expertise in Python, embeddings (Sentence Transformers, FAISS, Milvus, Pinecone, Qdrant).
- Proven track record deploying ML models to production with monitoring and rollback pipelines.
- Experience with NLP, semantic search, retrieval-augmented generation (RAG) a strong plus.
"""

def mms(series):
    s = series.fillna(0)
    return pd.Series(0.0, index=s.index) if s.max()==s.min() else (s-s.min())/(s.max()-s.min())

def build_technical_score(features):
    df = features.copy()
    ts = pd.DataFrame()
    ts["candidate_id"] = df["candidate_id"]
    ts["technical_score"] = (
        0.25*mms(df["ai_skill_score"])+0.25*mms(df["production_score"])+
        0.20*mms(df["career_quality_score"])+0.15*mms(df["python_endorsements"])+
        0.15*mms(df["embedding_skill_count"])
    )
    return ts

def build_product_fit_score(features):
    df = features.copy()
    ai_titles = ["AI Engineer","Applied Scientist","Machine Learning","ML Engineer",
                 "Recommendation","Search","Retrieval","Ranking","NLP"]
    tm = df["current_title"].str.contains("|".join(ai_titles), case=False, na=False)
    score = tm.astype(int)*0.35+mms(df["product_company_ratio"])*0.35+mms(df["retrieval_mentions"])*0.15+mms(df["vector_db_skill_count"])*0.15-mms(df["service_company_ratio"])*0.10
    return pd.DataFrame({"candidate_id": df["candidate_id"], "product_fit_score": np.clip(score,0,1)})

def build_experience_fit_score(features):
    df = features.copy()
    exp = df["years_of_experience"]
    score = np.where((exp>=6)&(exp<=8),1.0, np.where((exp>=5)&(exp<=9),0.9, np.where((exp>=4)&(exp<=10),0.7,0.3)))
    return pd.DataFrame({"candidate_id": df["candidate_id"], "experience_fit_score": score})

def build_final_score(ranking_df, technical_score, product_fit, experience_fit):
    df = ranking_df.merge(technical_score,on="candidate_id").merge(product_fit,on="candidate_id").merge(experience_fit,on="candidate_id")
    for col in ["semantic_similarity","technical_score","behavior_score","product_fit_score","experience_fit_score"]:
        df[col] = mms(df[col])
    df["final_score"] = (0.35*df["semantic_similarity"]+0.30*df["technical_score"]+
                         0.15*df["behavior_score"]+0.10*df["product_fit_score"]+0.10*df["experience_fit_score"])
    df = df.drop(columns=["trust_score_x"], errors="ignore")
    df = df.rename(columns={"trust_score_y": "trust_score"}, errors="ignore")
    return df

def rank_candidates(df):
    return df.sort_values("final_score", ascending=False).reset_index(drop=True)

def build_candidate_documents(candidates):
    docs = []
    for c in candidates:
        p = c["profile"]
        skills = ", ".join(s["name"] for s in c["skills"])
        career = " ".join(j["description"] for j in c["career_history"])
        text = f"Headline:\n{p['headline']}\n\nSummary:\n{p['summary']}\n\nCurrent Title:\n{p['current_title']}\n\nYears of Experience:\n{p['years_of_experience']}\n\nSkills:\n{skills}\n\nCareer History:\n{career}"
        docs.append({"candidate_id": c["candidate_id"], "text": text})
    return pd.DataFrame(docs)

def compute_semantic_scores(model, jd, docs):
    print("  Encoding job description...")
    jd_emb = model.encode(jd, normalize_embeddings=True, convert_to_numpy=True)
    print(f"  Encoding {len(docs)} candidate documents...")
    embs = model.encode(docs["text"].tolist(), normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    sim = cos_sim(embs, jd_emb.reshape(1,-1)).flatten()
    return pd.DataFrame({"candidate_id": docs["candidate_id"].values, "semantic_similarity": sim})

print('Ranking + Embedding modules loaded.')
print()
print('All source modules ready! Proceed to Step 3.')

---
## Step 3 — Upload Your Candidate File

Upload **any valid `candidates.jsonl`** file (one JSON object per line).

The small **`candidte.jsonl`** (2 records from repo root) is perfect for a quick test.  
Any valid subset of `candidates.jsonl` also works.

Expected per-record schema: `candidate_id`, `profile`, `skills`, `career_history`, `education`, `redrob_signals`

In [ ]:
# STEP 3: Upload candidate .jsonl file
from google.colab import files

print('Please upload your candidates .jsonl file below...')
uploaded = files.upload()

if not uploaded:
    raise ValueError('No file uploaded. Please re-run this cell and upload a file.')

filename = list(uploaded.keys())[0]
content  = uploaded[filename].decode('utf-8')

candidates = load_candidates_from_string(content)

print(f"\nFile '{filename}' uploaded successfully!")
print(f"Total candidates loaded: {len(candidates)}")

# Schema check
required_keys = {'candidate_id', 'profile', 'skills', 'career_history', 'redrob_signals'}
missing = required_keys - set(candidates[0].keys())
if missing:
    print(f"WARNING: First record is missing keys: {missing}")
else:
    print('Schema validation passed — all required top-level keys present.')

# Preview first candidate
c = candidates[0]
print(f"\nPreview of first candidate:")
print(f"  ID     : {c['candidate_id']}")
print(f"  Name   : {c['profile'].get('anonymized_name','N/A')}")
print(f"  Title  : {c['profile'].get('current_title','N/A')}")
print(f"  Exp    : {c['profile'].get('years_of_experience','N/A')} years")
print(f"  Skills : {len(c['skills'])} listed")
print(f"  Jobs   : {len(c['career_history'])} career roles")

---
## Step 4 — Preprocess & Build Features

In [ ]:
# STEP 4: Preprocessing + 60+ feature engineering
print('Preprocessing candidate data...')
data = prepare_data(candidates)
print(f"  Profiles : {len(data['profiles'])} rows")
print(f"  Skills   : {len(data['skills'])} rows")
print(f"  Career   : {len(data['career'])} rows")
print(f"  Signals  : {len(data['signals'])} rows")

print('\nBuilding feature table (60+ features)...')
features = build_all_features(
    profiles_df = data['profiles'],
    career_df   = data['career'],
    skill_df    = data['skills'],
    signals_df  = data['signals']
)
print(f"  Feature table: {features.shape[0]} candidates x {features.shape[1]} features")
features.head()

---
## Step 5 — Semantic Scoring (BAAI/bge-small-en-v1.5)
Model download ~130 MB, takes ~30 sec first time.

In [ ]:
# STEP 5: Semantic similarity scoring against the job description
print('Loading embedding model: BAAI/bge-small-en-v1.5 ...')
model = SentenceTransformer('BAAI/bge-small-en-v1.5')
print('Model loaded!')

print('\nBuilding candidate semantic documents...')
documents = build_candidate_documents(candidates)

print('\nComputing cosine similarities...')
semantic_scores = compute_semantic_scores(model, JOB_DESCRIPTION, documents)

print('\nSemantic scoring complete!')
semantic_scores

---
## Step 6 — Behavior Scoring

In [ ]:
# STEP 6: Compute behavior scores
print('Computing behavior scores...')
behavior_scores = build_behavior_score(data['signals'])
print(f"  Shape: {behavior_scores.shape}")
behavior_scores[['candidate_id','availability_score','recruiter_interest_score','trust_score','engagement_score','behavior_score']]

---
## Step 7 — Final Ranking

**Weights:** 35% semantic · 30% technical · 15% behavior · 10% product fit · 10% experience fit

In [ ]:
# STEP 7: Weighted ranking
print('Building final ranking...')

behavior_cols = ['candidate_id','availability_score','recruiter_interest_score','trust_score','engagement_score','behavior_score']
ranking_df = features.merge(semantic_scores, on='candidate_id').merge(behavior_scores[behavior_cols], on='candidate_id')

technical  = build_technical_score(features)
product    = build_product_fit_score(features)
experience = build_experience_fit_score(features)

ranking_df = build_final_score(ranking_df, technical, product, experience)
ranking_df = rank_candidates(ranking_df)
ranking_df['rank'] = range(1, len(ranking_df)+1)

print(f"Ranked {len(ranking_df)} candidate(s).")

score_cols = ['candidate_id','rank','current_title','years_of_experience',
              'semantic_similarity','technical_score','behavior_score',
              'product_fit_score','experience_fit_score','final_score']
ranking_df[[c for c in score_cols if c in ranking_df.columns]]

---
## Step 8 — Generate Natural-Language Reasoning

In [ ]:
# STEP 8: Generate reasoning for each candidate
def generate_reasoning(row):
    r = []
    r.append(f"{row.get('years_of_experience',0):.1f} years as {row.get('current_title','Unknown')}")
    sem = row.get('semantic_similarity', 0)
    if sem >= 0.85:   r.append('excellent semantic match to the AI Engineer job description')
    elif sem >= 0.75: r.append('strong semantic match to the required profile')
    elif sem >= 0.60: r.append('moderate semantic match to the job description')
    if row.get('retrieval_mentions',0) > 0:      r.append('demonstrates retrieval or ranking experience')
    if row.get('embedding_skill_count',0) > 0:   r.append('has embeddings experience')
    if row.get('vector_db_skill_count',0) > 0:   r.append('worked with vector databases')
    if row.get('product_company_ratio',0) >= 0.50:   r.append('strong product-company background')
    elif row.get('service_company_ratio',0) >= 0.80: r.append('primarily service-company experience')
    tech = row.get('technical_score', 0)
    if tech >= 0.80:   r.append('excellent technical strength')
    elif tech >= 0.65: r.append('solid technical profile')
    beh = row.get('behavior_score', 0)
    if beh >= 0.80:   r.append('high recruiter engagement and availability')
    elif beh >= 0.60: r.append('good recruiter engagement')
    if row.get('years_of_experience', 10) < 5: r.append('slightly below the preferred experience range')
    return '. '.join(r) + '.'

ranking_df['reasoning'] = ranking_df.apply(generate_reasoning, axis=1)

print('Reasoning generated for all candidates.')
print()
for _, row in ranking_df[['candidate_id','rank','final_score','reasoning']].iterrows():
    print(f"Rank #{int(row['rank'])} | {row['candidate_id']} | Score: {row['final_score']:.4f}")
    print(f"   {row['reasoning']}")
    print()

---
## Step 9 — Validate & Download Submission CSV

In [ ]:
# STEP 9: Build submission CSV, validate, and download
submission = ranking_df[['candidate_id','rank','final_score','reasoning']].copy()
submission.rename(columns={'final_score':'score'}, inplace=True)

n = len(submission)

# --- Validation checks ---
assert n >= 1,                                          'No candidates in submission!'
assert submission['candidate_id'].nunique() == n,       'Duplicate candidate IDs found!'
assert submission['rank'].tolist() == list(range(1,n+1)),'Ranks are not sequential 1..n!'
assert submission['score'].is_monotonic_decreasing,     'Scores are not sorted descending!'
assert submission['reasoning'].notna().all(),           'Some candidates missing reasoning!'

print('Submission validated successfully!')
print(f"  Candidates ranked : {n}")
print(f"  Unique IDs        : {submission['candidate_id'].nunique()}")
print(f"  Score range       : {submission['score'].min():.4f} to {submission['score'].max():.4f}")
print(f"  Scores descending : Yes")
print()
print(submission.to_string(index=False))

# --- Save and download ---
TEAM_ID = 'UniverseX'
output_path = f'/content/{TEAM_ID}_sandbox_submission.csv'
submission.to_csv(output_path, index=False, encoding='utf-8')
print(f"\nSaved: {output_path}")

files.download(output_path)
print('Download started! Check your downloads folder.')

print()
print('=' * 60)
print('PIPELINE COMPLETE — All steps ran successfully!')
print('=' * 60)
print('This sandbox uses the IDENTICAL pipeline as our final')
print('submission — same source modules, same weights, same logic.')

---
## Appendix — Pipeline Architecture

```
candidates.jsonl
      |
      v
[preprocessing]  ->  profiles_df, skills_df, career_df, signals_df
      |
      +---------------------------+
      v                           v
[feature_engineering]      [embedding.py]
  60+ features              BAAI/bge-small-en-v1.5
  career/skill/             cosine_similarity vs JD
  behavior                         |
      |          [behavior_scoring]|
      |          availability      |
      |          trust             |
      |          engagement        |
      |          recruiter_interest|
      |                |           |
      +----------------+-----------+
                       v
              [ranking.py]
              35% semantic_similarity
              30% technical_score
              15% behavior_score
              10% product_fit_score
              10% experience_fit_score
                       |
                       v
         UniverseX_sandbox_submission.csv
       candidate_id | rank | score | reasoning
```

| Score | Weight | Captures |
|-------|--------|----------|
| Semantic Similarity | 35% | JD-to-resume embedding cosine distance |
| Technical Score | 30% | AI skill depth, production experience, career quality |
| Behavior Score | 15% | Availability, trust, engagement, recruiter interest |
| Product Fit | 10% | Product-company background, title match, retrieval/vector DB |
| Experience Fit | 10% | Years of experience relative to JD range (5-9 yrs optimal) |

---
*Team UniverseX · Redrob Ranker Challenge*